# Cross-species embedding — scPRINT `te0uwaz1`

Objective: run checkpoint `te0uwaz1.ckpt` on the shared cat/tiger
benchmark after both species were remapped to mouse genes, then compute the
same scIB scores as the original notebook.

The input organism is forced to mouse (`NCBITaxon:10090`), so scPRINT uses
the checkpoint's mouse gene embeddings for every cell.

In [1]:
import os
from pathlib import Path

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "HTTP_PROXY": "http://127.0.0.1:9",
    "HTTPS_PROXY": "http://127.0.0.1:9",
    "ALL_PROXY": "http://127.0.0.1:9",
    "NO_PROXY": "",
})

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

SEED = 42
rng = np.random.default_rng(SEED)

DATA_PATH = Path(
    "notebooks/scPRINT-2-repro-notebooks/data/task_3_embed.h5ad"
)
RESULT_ROOT = Path("data/results/cross_species_embedding")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_KEY = "orig.ident"
LABEL_KEY = "NewCelltype"
MOUSE_ONTOLOGY_ID = "NCBITaxon:10090"

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

import torch
import scdataloader.collator as scd_collator
from scprint2 import scPRINT2
from scprint2.model import utils as scprint_model_utils
from scprint2.tasks import Embedder

CHECKPOINT_PATH = Path("/lustre/fswork/projects/rech/xeg/uat95fg/te0uwaz1.ckpt")
OUTPUT_PATH = RESULT_ROOT / "scprint_te0uwaz1_embeddings.h5ad"
SCORE_PATH = RESULT_ROOT / "scprint_te0uwaz1_scib.csv"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a Slurm GPU allocation")

torch.set_float32_matmul_precision("medium")

→ connected lamindb: jkobject/scprint2


/lustre/fswork/projects/rech/xeg/uat95fg/simpler_flash/src/simpler_flash/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/lustre/fswork/projects/rech/xeg/uat95fg/simpler_flash/src/simpler_flash/layer_norm.py:1107: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


## Load the remapped mouse input and checkpoint

The dataset was produced by the original notebook's mouse-remapping option.
No cat- or tiger-specific gene embeddings are added here.

In [2]:
da = sc.read_h5ad(DATA_PATH)
da.obs["organism_ontology_term_id"] = MOUSE_ONTOLOGY_ID
if set(da.obs["organism_ontology_term_id"].astype(str)) != {MOUSE_ONTOLOGY_ID}:
    raise ValueError("All cells must use the mouse organism token")

model = scPRINT2.load_from_checkpoint(
    CHECKPOINT_PATH,
    precpt_gene_emb=None,
    gene_pos_file=None,
    map_location="cpu",
)

mouse_genes = list(model._genes[MOUSE_ONTOLOGY_ID])
missing_mouse_genes = set(mouse_genes) - set(da.var_names)
if missing_mouse_genes:
    raise ValueError(
        f"Input is missing {len(missing_mouse_genes)} checkpoint mouse genes"
    )
da = da[:, mouse_genes].copy()

def checkpoint_gene_table(organisms):
    """Build Collator metadata only from genes stored in the checkpoint."""
    if not isinstance(model._genes, dict):
        raise TypeError("Expected a checkpoint-embedded gene dictionary")
    if isinstance(organisms, str):
        organisms = [organisms]
    frames = [
        pd.DataFrame(
            {"organism": organism},
            index=pd.Index(model._genes[organism], name="ensembl_gene_id"),
        )
        for organism in organisms
    ]
    return pd.concat(frames)


def skip_ontology_translation(values, class_name):
    """Keep checkpoint ontology IDs without querying Bionty."""
    return None


scd_collator.load_genes = checkpoint_gene_table
scprint_model_utils.translate = skip_ontology_translation
model = model.to("cuda").eval()
print("Model organisms:", model.organisms)

FYI: scPRINT2 is not attached to a `Trainer`.


Model organisms: ['NCBITaxon:10090', 'NCBITaxon:9606']


## Generate scPRINT cell embeddings

In [3]:
embedder = Embedder(
    how="random expr",
    max_len=3200,
    num_workers=8,
    doclass=False,
    pred_embedding=["all"],
    doplot=False,
)
output_da, embedding_metrics = embedder(model, da.copy())
if "scprint_emb" not in output_da.obsm:
    raise KeyError("Embedder output has no obsm['scprint_emb']")
output_da.obsm["model_emb"] = np.asarray(output_da.obsm["scprint_emb"], dtype=np.float32)
output_da.write_h5ad(OUTPUT_PATH, compression="lzf")
output_da

predict epoch start


  0%|          | 0/425 [00:00<?, ?it/s]

  0%|          | 1/425 [00:08<59:02,  8.35s/it]

  0%|          | 2/425 [00:08<25:49,  3.66s/it]

  1%|          | 3/425 [00:09<15:13,  2.16s/it]

...

logging the anndata


AnnData object with n_obs × n_vars = 27200 × 0
    obs: 'pred_cell_type_ontology_term_id', 'pred_tissue_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id'
    obsm: 'scprint_emb_other', 'scprint_emb_cell_type_ontology_term_id', 'scprint_emb_tissue_ontology_term_id', 'scprint_emb_disease_ontology_term_id', 'scprint_emb_assay_ontology_term_id', 'scprint_emb_self_reported_ethnicity_ontology_term_id', 'scprint_emb_sex_ontology_term_id', 'scprint_emb_organism_ontology_term_id'


... storing 'organism_ontology_term_id' as categorical


too few cells to embed into a umap
too few cells to compute a clustering


AnnData object with n_obs × n_vars = 27200 × 21598
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'cell_type', 'batch', 'barcode', 'celltype', 'percent.mt', 'integrated_snn_res.1', 'NewCelltype', 'n_genes', 'organism_ontology_term_id', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'pred_cell_type_ontology_term_id', 'pred_tissue_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id'
    var: 'uid', 'symbol', 'biotype', 'organism_id', 'branch_id', 'mt', 'ribo', 'hb', 'organism', 'ensemb

## scIB embedding scores

In [4]:
def add_common_baselines(
    adata: ad.AnnData,
    model_embedding_key: str,
    expression_source: ad.AnnData | None = None,
) -> None:
    """Add comparable PCA and seeded-random embeddings without changing raw counts."""
    source = adata if expression_source is None else expression_source
    if not source.obs_names.equals(adata.obs_names):
        raise RuntimeError("PCA source cell names/order differ from model output")
    baseline = source.copy()
    sc.pp.normalize_total(baseline, target_sum=1e4)
    sc.pp.log1p(baseline)
    sc.pp.pca(baseline, n_comps=50)
    adata.obsm["X_pca"] = np.asarray(baseline.obsm["X_pca"], dtype=np.float32)
    adata.obsm["random"] = rng.random(adata.obsm["X_pca"].shape, dtype=np.float32)
    if model_embedding_key not in adata.obsm:
        raise KeyError(f"Missing model embedding: {model_embedding_key}")


def benchmark_embeddings(adata: ad.AnnData, model_embedding_key: str):
    """Run the original cross-species scIB comparison and return unscaled scores."""
    for key in (BATCH_KEY, LABEL_KEY):
        if key not in adata.obs:
            raise KeyError(f"Missing obs column: {key}")
    benchmark = Benchmarker(
        adata,
        batch_key=BATCH_KEY,
        label_key=LABEL_KEY,
        embedding_obsm_keys=[model_embedding_key, "X_pca", "random"],
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        n_jobs=10,
    )
    benchmark.benchmark()
    return benchmark.get_results(min_max_scale=False)

In [5]:
add_common_baselines(output_da, "model_emb")
results = benchmark_embeddings(output_da, "model_emb")
results.to_csv(SCORE_PATH)
output_da.write_h5ad(OUTPUT_PATH, compression="lzf")
display(results)
print("Scores:", SCORE_PATH)

/lustre/fswork/projects/rech/xeg/uat95fg/scPRINT/.venv/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


Computing neighbors:   0%|          | 0/3 [00:00<?, ?it/s]

Computing neighbors:  33%|███▎      | 1/3 [00:17<00:34, 17.39s/it]

Computing neighbors:  67%|██████▋   | 2/3 [00:25<00:11, 11.77s/it]

Computing neighbors: 100%|██████████| 3/3 [00:29<00:00,  8.30s/it]

...

Metrics:  90%|█████████ | 9/10 [00:52<00:04,  4.37s/it, Batch correction: pcr_comparison]

Embeddings: 100%|██████████| 3/3 [04:55<00:00, 83.40s/it] 

Embeddings: 100%|██████████| 3/3 [04:55<00:00, 98.51s/it]

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,BRAS,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
model_emb,0.526165,0.242519,0.163367,0.473985,0.948391,0.622032,0.0,0.008182,0.659853,0,0.258013,0.470885,0.385737
X_pca,0.577403,0.214784,0.114595,0.382667,0.996476,0.523025,0.0,0.000386,0.563837,0.0,0.217449,0.457185,0.361291
random,0.491522,0.000959,-0.000003,0.487868,0.675803,0.994207,0.858965,0.961152,0.200675,0.999575,0.802915,0.33123,0.519904
Metric Type,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Batch correction,Batch correction,Batch correction,Batch correction,Batch correction,Aggregate score,Aggregate score,Aggregate score


Scores: data/results/cross_species_embedding/scprint_te0uwaz1_scib.csv
